In [1]:
import torch
from diffusers import (
    AutoencoderKLCogVideoX, 
    CogVideoXPipeline, 
    CogVideoXTransformer3DModel
)
from diffusers.utils import export_to_video
from transformers import T5EncoderModel

/Users/alexeyfilichkin/Desktop/ALPHA/SAT/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [29]:
DEVICE = (torch.device('mps') if torch.backends.mps.is_available()
          else torch.device('cpu'))
print(f'Training on device {DEVICE}')

Training on device mps


In [30]:
model_id = './CogVideo/CogVideoX1.5-5B'

In [31]:
vae = AutoencoderKLCogVideoX.from_pretrained(
    model_id, 
    subfolder='vae', 
    torch_dtype=torch.float16
    )

In [32]:
transformer = CogVideoXTransformer3DModel.from_pretrained(
    './CogVideo/cogvideox-5b-float16', 
    subfolder='transformer', 
    torch_dtype=torch.float16
    )
text_encoder = T5EncoderModel.from_pretrained(
    './CogVideo/cogvideox-5b-float16', 
    subfolder='text_encoder', 
    torch_dtype=torch.float16
    )


Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00,  8.64it/s]


In [33]:
pipe = CogVideoXPipeline.from_pretrained(
    model_id,
    text_encoder=text_encoder,
    transformer=transformer,
    vae=vae,
    torch_dtype=torch.float16,
)

Loading pipeline components...: 100%|██████████| 5/5 [00:00<00:00, 37.36it/s]


In [34]:
pipe

CogVideoXPipeline {
  "_class_name": "CogVideoXPipeline",
  "_diffusers_version": "0.32.2",
  "_name_or_path": "./CogVideo/CogVideoX1.5-5B",
  "scheduler": [
    "diffusers",
    "CogVideoXDDIMScheduler"
  ],
  "text_encoder": [
    "transformers",
    "T5EncoderModel"
  ],
  "tokenizer": [
    "transformers",
    "T5Tokenizer"
  ],
  "transformer": [
    "diffusers",
    "CogVideoXTransformer3DModel"
  ],
  "vae": [
    "diffusers",
    "AutoencoderKLCogVideoX"
  ]
}

In [35]:
pipe.enable_sequential_cpu_offload()

In [ ]:
prompt = (
    'A panda, dressed in a small, red jacket and a tiny hat, sits on a wooden stool in a serene bamboo forest.'
)

In [ ]:
video = pipe(
    prompt=prompt, 
    guidance_scale=6, 
    use_dynamic_cfg=True, 
    num_inference_steps=50
    ).frames[0]

In [ ]:
export_to_video(video, './output2.mp4', fps=8)